In [ ]:

import json
import time
import urllib.request
import urllib.error
from pathlib import Path

API_URL = "http://localhost:11434/api/chat"

# Replace these with the exact model tags from `ollama list`.
MODELS = [
    "hf.co/abenzerps/Nemotron-3.5-Bolt-E4B-GSQ-RCO-GGUF:Q5_K_M",
    # Add the other model tags here.
]

with open("benchmark_request.json", "r", encoding="utf-8") as f:
    template = json.load(f)

out_dir = Path("benchmark_results")
out_dir.mkdir(exist_ok=True)

for model in MODELS:
    payload = {
        "model": model,
        "messages": template["messages"],
        "stream": False,
        "format": "json",
        "think": False,
        "options": {
            **template.get("options", {}),
            "temperature": 0,
            "num_predict": 8192
        }
    }

    started = time.perf_counter()

    try:
        request = urllib.request.Request(
            API_URL,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST"
        )

        with urllib.request.urlopen(request, timeout=1800) as response:
            result = json.loads(response.read().decode("utf-8"))

        elapsed = time.perf_counter() - started
        content = result.get("message", {}).get("content", "")

        try:
            parsed_json = json.loads(content)
            valid_json = True
        except (json.JSONDecodeError, TypeError):
            parsed_json = None
            valid_json = False

        record = {
            "model": model,
            "elapsed_seconds": round(elapsed, 3),
            "valid_json": valid_json,
            "response": parsed_json if valid_json else content,
            "ollama_total_duration_ns": result.get("total_duration"),
            "prompt_eval_count": result.get("prompt_eval_count"),
            "eval_count": result.get("eval_count"),
            "eval_duration_ns": result.get("eval_duration")
        }

    except Exception as exc:
        record = {
            "model": model,
            "error": str(exc)
        }

    filename = model.replace("/", "_").replace(":", "_") + ".json"
    (out_dir / filename).write_text(
        json.dumps(record, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print(model, "->", record.get("elapsed_seconds", "ERROR"),
          "seconds; valid JSON:", record.get("valid_json", False))

print(f"Results saved in: {out_dir.resolve()}")
